# Week 2, Day 3: A streaming shop assistant for replica titles

**Lesson:** [week2/day3.ipynb](../../../week2/day3.ipynb), *Conversational AI: a chatbot*. **About 35 minutes.**

**What you practice here**

- The callback `gr.ChatInterface` calls, `chat(message, history)`, and its messages built from Gradio's history: the system message first, the history with only `role` and `content`, then the new user message.
- Calling the model from the callback, then streaming its reply and yielding it as it grows.
- Adding context to the system message for one turn only, when the customer's message calls for it, as the lesson does for belts.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week2/day3", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson builds a chatbot for a clothes store: a system message that sets the scene and gives an example reply, a
callback that `gr.ChatInterface` calls with the new message and the history, and an extra line in the system message,
for one turn, when the customer asks about belts. Build the same for a store of your own that sells replica WWE
championship titles.

The two callbacks that call the model, `chat` and `stream_chat`, take the client as their first parameter and make
the Chat Completions call themselves. In Try it for real, a two-line function hands Gradio the `chat(message, history)`
it expects. The checks pass a stand-in shaped like the real client, so nothing is spent.

## What each call should give

```python
to_messages("You help shoppers.",
            [{"role": "user", "content": "Do you sell an Intercontinental Championship replica?",
              "metadata": None, "options": None},
             {"role": "assistant", "content": "Yes, we do.", "metadata": None, "options": None}],
            "Is it full size?")
# [{'role': 'system', 'content': 'You help shoppers.'},
#  {'role': 'user', 'content': 'Do you sell an Intercontinental Championship replica?'},
#  {'role': 'assistant', 'content': 'Yes, we do.'},
#  {'role': 'user', 'content': 'Is it full size?'}]

system_for("Do you have AUTOGRAPHED titles?", "You help shoppers.")
# 'You help shoppers.\nThe store sells no autographed items. If asked, suggest a replica title instead.'

chat(client, "Is it full size?", history)
# makes one request with the model gpt-4.1-mini and the messages from
# to_messages(system_for(message, SYSTEM), history, message), and returns the reply's text

list(stream_chat(client, "Is it full size?", history))
# makes the same request with streaming on, and yields the reply so far after each chunk: ['', 'Yes,', 'Yes, it is', ...]
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `SYSTEM`

Your store assistant's system message: what the store sells, and how the assistant should answer. Like the lesson's
clothes-store message, say what is on sale (such as a discount on one title), and give one example of a customer's
line with a good reply: that is one-shot prompting. Write it as a string of 20 characters or more.

<details><summary>Hint</summary>

A triple-quoted string, like the lesson's: who the assistant is and what is on sale, then
`For example, if the customer says '...', you could reply '...'`.

</details>

In [ ]:
SYSTEM = ""

In [ ]:
check("SYSTEM")

### 2 · `to_messages(system, history, message)`

Builds the messages for one turn from Gradio's history. Return a new list: a system message holding `system`, then
one message for each history entry, in order, with that entry's role and content, then a user message holding
`message`. Gradio's history entries can carry more keys, such as `"metadata"` and `"options"`: copy only `"role"` and
`"content"`, as the lesson does. Do not change the history you were given, nor its entries.

<details><summary>Hint</summary>

The lesson's two lines:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
```

</details>

In [ ]:
def to_messages(system, history, message):
    # Write to_messages here, as described above.
    raise NotImplementedError("to_messages is not written yet")

In [ ]:
check("to_messages")

### 3 · `system_for(message, base)`

Adds a line to the system message, for this turn only, when the customer asks about autographs. Like the lesson's
check for `'belt'`, it returns `base` unchanged unless `message` contains `autograph` in any mix of upper and lower
case, so `"AUTOGRAPHED"` counts too. When it does, it returns `base`, a new line, and exactly this line:
`The store sells no autographed items. If asked, suggest a replica title instead.`

<details><summary>Hint</summary>

`if "autograph" in message.lower():`, as the lesson tests for `'belt'`, then join with `"\n"`.

</details>

In [ ]:
def system_for(message, base):
    # Write system_for here, as described above.
    raise NotImplementedError("system_for is not written yet")

In [ ]:
check("system_for")

### 4 · `chat(client, message, history, model="gpt-4.1-mini")`

Answers one message in one call, and returns the reply's text. Build the messages with `to_messages`: the system
message from `system_for` (for this message, on top of `SYSTEM`), the history, and the message. Make one Chat
Completions call with the model `model` and those messages, and return the reply's text. Do not change the history.

<details><summary>Hint</summary>

```python
messages = to_messages(system_for(message, SYSTEM), history, message)
response = client.chat.completions.create(model=model, messages=messages)
return response.choices[0].message.content
```

</details>

In [ ]:
def chat(client, message, history, model="gpt-4.1-mini"):
    # Write chat here, as described above.
    raise NotImplementedError("chat is not written yet")

In [ ]:
check("chat")

### 5 · `stream_chat(client, message, history, model="gpt-4.1-mini")`

Answers one message with streaming, and yields the reply as it grows. Build the same messages as `chat`, and make one
Chat Completions call with streaming on. For each chunk, add its text to the reply so far (counting a `None` text as
`""`), and yield the reply so far, as the lesson's streaming `chat` does. Change neither `SYSTEM` nor the history: the
autograph line belongs to this turn only, like the lesson's `relevant_system_message`.

<details><summary>Hint</summary>

```python
stream = client.chat.completions.create(model=model, messages=messages, stream=True)
response = ""
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    yield response
```

</details>

In [ ]:
def stream_chat(client, message, history, model="gpt-4.1-mini"):
    # Write stream_chat here, as described above.
    raise NotImplementedError("stream_chat is not written yet")

In [ ]:
check("stream_chat")

## Check everything

The cell below runs all 9 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 2, Day 3 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Each answer from `gpt-4.1-mini` costs a fraction of a cent.

1. Load your key with `load_dotenv(override=True)`, and make `openai = OpenAI()`.
2. Write a generator function `respond(message, history)` that does `yield from stream_chat(openai, message, history)`.
   Gradio streams only from a generator function, so a lambda will not do.
3. Run `gr.ChatInterface(fn=respond, type="messages").launch()`, after `import gradio as gr`. Ask about a title, then
   ask for an autographed one.
4. Optional: the same store on a local model. With Ollama running, pass
   `OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")` and `model="llama3.2"` to `stream_chat`.

<details><summary>Stuck? One way to write it</summary>

```python
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai = OpenAI()


def respond(message, history):  # a generator function, so the chat streams
    yield from stream_chat(openai, message, history)


gr.ChatInterface(fn=respond, type="messages").launch()
```

```python
# Optional: the same store on a local model, with Ollama running
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


def respond_locally(message, history):
    yield from stream_chat(ollama, message, history, model="llama3.2")


gr.ChatInterface(fn=respond_locally, type="messages").launch()
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.